# SPY 5-Minute Signal Arrows

A clean SPY 5-minute candlestick chart with **only buy/sell arrows** on it — no
overlays, no sub-panels. Signals are still generated from the full engine
underneath: EMA/MACD/RSI/VWAP/Bollinger indicators, clustered support/resistance,
sloped trend lines, classic chart patterns (double top/bottom, head & shoulders,
triangles) and harmonic patterns (Gartley, Bat, Butterfly, Crab) — it's just not
drawn on the chart itself, to keep it readable.

Signals are computed **causally, bar by bar** (each bar only "knows" what happened
before it), so the arrows you see are exactly what the tool would have signaled in
real time — this also means the same code works for a historical replay and for
the live/current bar.

**How to use in Google Colab**
1. Run the *Install packages* cell once per session.
2. Run every cell top to bottom once.
3. Re-run just the **last cell** (`run_analysis()`) as often as you like during the
   trading day — each call re-fetches fresh data and redraws the chart for the
   current session with up-to-date arrows.
4. Or run the *Live loop* cell to have it auto-refresh every few minutes on its own.

**Disclaimer:** Educational tool only, not financial advice. Free `yfinance` data
can lag real-time by up to ~15 minutes. Always validate signals yourself before
trading.


## 1. Install packages

In [ ]:
!pip install -q yfinance plotly scipy


## 2. Imports & configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.signal import argrelextrema
import plotly.graph_objects as go

# ---- Configuration ----
SYMBOL = "SPY"
INTERVAL = "5m"        # 5-minute candles
PERIOD = "5d"          # extra days give the engine context (pivots/patterns/trend lines);
                       # the chart itself only shows the current session (see run_analysis)
PIVOT_WINDOW = 5              # bars each side required to confirm a swing high/low
SR_TOLERANCE_PCT = 0.0015     # cluster tolerance for horizontal support/resistance (0.15%)
HARMONIC_TOLERANCE = 0.07     # tolerance band around ideal Fibonacci ratios
TRENDLINE_LOOKBACK = 6        # how many recent swing points to fit each trend line on
CAUSAL_LOOKBACK = 150         # bars of history the engine looks back over at each step
WARMUP_BARS = 60              # bars needed before the first signal can be computed


## 3. Data fetch

In [ ]:
def fetch_data(symbol=SYMBOL, interval=INTERVAL, period=PERIOD):
    df = yf.download(symbol, interval=interval, period=period, progress=False, auto_adjust=False)
    if df.empty:
        raise ValueError("No data returned — market may be closed, symbol invalid, or rate-limited.")
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index.name = "Datetime"
    if df.index.tz is None:
        df = df.tz_localize("UTC")
    df = df.tz_convert("America/New_York")
    df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
    return df


## 4. Indicators (EMA, MACD, RSI, Bollinger Bands, ATR, VWAP)

In [ ]:
def add_indicators(df):
    df = df.copy()

    df["EMA9"] = df["Close"].ewm(span=9, adjust=False).mean()
    df["EMA21"] = df["Close"].ewm(span=21, adjust=False).mean()
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()

    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_hist"] = df["MACD"] - df["MACD_signal"]

    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / 14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / 14, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    df["RSI"] = 100 - (100 / (1 + rs))
    df["RSI"] = df["RSI"].fillna(50)

    sma20 = df["Close"].rolling(20).mean()
    std20 = df["Close"].rolling(20).std()
    df["BB_mid"] = sma20
    df["BB_upper"] = sma20 + 2 * std20
    df["BB_lower"] = sma20 - 2 * std20

    high_low = df["High"] - df["Low"]
    high_close = (df["High"] - df["Close"].shift()).abs()
    low_close = (df["Low"] - df["Close"].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["ATR"] = tr.ewm(alpha=1 / 14, adjust=False).mean()

    # Session VWAP — resets every trading day. All of the above are rolling/ewm
    # (backward-looking only) so every value here is causal: it only uses bars
    # up to and including its own row.
    session_date = df.index.date
    typical = (df["High"] + df["Low"] + df["Close"]) / 3
    tpv = typical * df["Volume"]
    df["VWAP"] = tpv.groupby(session_date).cumsum() / df["Volume"].groupby(session_date).cumsum()

    return df


## 5. Swing pivots, support/resistance clusters, trend lines

In [ ]:
def find_pivots(df, window=PIVOT_WINDOW):
    highs = df["High"].values
    lows = df["Low"].values
    piv_high_idx = argrelextrema(highs, np.greater, order=window)[0]
    piv_low_idx = argrelextrema(lows, np.less, order=window)[0]
    return sorted(piv_high_idx.tolist()), sorted(piv_low_idx.tolist())


def cluster_levels(df, piv_idx, price_col, tolerance_pct=SR_TOLERANCE_PCT, min_touches=2, max_levels=6):
    prices = df[price_col].values[piv_idx]
    if len(prices) == 0:
        return []
    levels = []
    for p in sorted(prices):
        placed = False
        for lvl in levels:
            if abs(p - lvl["price"]) / lvl["price"] <= tolerance_pct:
                lvl["prices"].append(p)
                lvl["price"] = float(np.mean(lvl["prices"]))
                lvl["touches"] += 1
                placed = True
                break
        if not placed:
            levels.append({"price": float(p), "prices": [p], "touches": 1})
    levels = [lvl for lvl in levels if lvl["touches"] >= min_touches]
    levels.sort(key=lambda x: -x["touches"])
    return levels[:max_levels]


def get_support_resistance(df, piv_high_idx, piv_low_idx):
    resistance = cluster_levels(df, piv_high_idx, "High")
    support = cluster_levels(df, piv_low_idx, "Low")
    return support, resistance


def fit_trendline(idx_list, price_list, n_recent=TRENDLINE_LOOKBACK):
    if len(idx_list) < 2:
        return None
    idx_arr = np.array(idx_list[-n_recent:], dtype=float)
    price_arr = np.array(price_list[-n_recent:], dtype=float)
    slope, intercept = np.polyfit(idx_arr, price_arr, 1)
    return float(slope), float(intercept)


def get_trendlines(df, piv_high_idx, piv_low_idx):
    resistance_line = None
    support_line = None
    if len(piv_high_idx) >= 2:
        resistance_line = fit_trendline(piv_high_idx, df["High"].values[piv_high_idx])
    if len(piv_low_idx) >= 2:
        support_line = fit_trendline(piv_low_idx, df["Low"].values[piv_low_idx])
    return support_line, resistance_line


## 6. Classic chart patterns (double top/bottom, head & shoulders, triangles)

In [ ]:
def detect_double_top_bottom(df, piv_high_idx, piv_low_idx, tolerance=0.002):
    patterns = []
    if len(piv_high_idx) >= 2:
        i1, i2 = piv_high_idx[-2], piv_high_idx[-1]
        p1, p2 = df["High"].iloc[i1], df["High"].iloc[i2]
        if abs(p1 - p2) / p1 <= tolerance:
            neckline = df["Low"].iloc[i1:i2 + 1].min()
            confirmed = bool(df["Close"].iloc[-1] < neckline)
            patterns.append({"pattern": "Double Top", "bias": "bearish",
                              "neckline": float(neckline), "index": i2, "confirmed": confirmed})
    if len(piv_low_idx) >= 2:
        i1, i2 = piv_low_idx[-2], piv_low_idx[-1]
        p1, p2 = df["Low"].iloc[i1], df["Low"].iloc[i2]
        if abs(p1 - p2) / p1 <= tolerance:
            neckline = df["High"].iloc[i1:i2 + 1].max()
            confirmed = bool(df["Close"].iloc[-1] > neckline)
            patterns.append({"pattern": "Double Bottom", "bias": "bullish",
                              "neckline": float(neckline), "index": i2, "confirmed": confirmed})
    return patterns


def detect_head_shoulders(df, piv_high_idx, piv_low_idx, tolerance=0.01):
    patterns = []
    if len(piv_high_idx) >= 3:
        i1, i2, i3 = piv_high_idx[-3:]
        p1, p2, p3 = df["High"].iloc[i1], df["High"].iloc[i2], df["High"].iloc[i3]
        if p2 > p1 and p2 > p3 and abs(p1 - p3) / p1 <= tolerance:
            neckline = df["Low"].iloc[i1:i3 + 1].min()
            confirmed = bool(df["Close"].iloc[-1] < neckline)
            patterns.append({"pattern": "Head & Shoulders", "bias": "bearish",
                              "neckline": float(neckline), "index": i3, "confirmed": confirmed})
    if len(piv_low_idx) >= 3:
        i1, i2, i3 = piv_low_idx[-3:]
        p1, p2, p3 = df["Low"].iloc[i1], df["Low"].iloc[i2], df["Low"].iloc[i3]
        if p2 < p1 and p2 < p3 and abs(p1 - p3) / p1 <= tolerance:
            neckline = df["High"].iloc[i1:i3 + 1].max()
            confirmed = bool(df["Close"].iloc[-1] > neckline)
            patterns.append({"pattern": "Inverse Head & Shoulders", "bias": "bullish",
                              "neckline": float(neckline), "index": i3, "confirmed": confirmed})
    return patterns


def detect_triangle(support_line, resistance_line, avg_price, flat_thresh_pct=0.0002):
    if support_line is None or resistance_line is None or avg_price == 0:
        return None
    s_slope_pct = support_line[0] / avg_price
    r_slope_pct = resistance_line[0] / avg_price
    if abs(r_slope_pct) <= flat_thresh_pct and s_slope_pct > flat_thresh_pct:
        return {"pattern": "Ascending Triangle", "bias": "bullish", "confirmed": False}
    if abs(s_slope_pct) <= flat_thresh_pct and r_slope_pct < -flat_thresh_pct:
        return {"pattern": "Descending Triangle", "bias": "bearish", "confirmed": False}
    if r_slope_pct < -flat_thresh_pct and s_slope_pct > flat_thresh_pct:
        return {"pattern": "Symmetrical Triangle", "bias": "neutral", "confirmed": False}
    return None


## 7. Harmonic patterns (Gartley, Bat, Butterfly, Crab)

In [ ]:
HARMONIC_DEFS = {
    "Gartley":   {"AB_XA": (0.618, 0.618), "BC_AB": (0.382, 0.886), "CD_BC": (1.13, 1.618), "AD_XA": (0.786, 0.786)},
    "Bat":       {"AB_XA": (0.382, 0.5),   "BC_AB": (0.382, 0.886), "CD_BC": (1.618, 2.618), "AD_XA": (0.886, 0.886)},
    "Butterfly": {"AB_XA": (0.786, 0.786), "BC_AB": (0.382, 0.886), "CD_BC": (1.618, 2.24),  "AD_XA": (1.27, 1.618)},
    "Crab":      {"AB_XA": (0.382, 0.618), "BC_AB": (0.382, 0.886), "CD_BC": (2.24, 3.618),  "AD_XA": (1.618, 1.618)},
}


def get_alternating_pivots(df, piv_high_idx, piv_low_idx):
    """Merge swing highs/lows into one time-ordered, strictly alternating sequence."""
    points = [(i, float(df["High"].iloc[i]), "H") for i in piv_high_idx]
    points += [(i, float(df["Low"].iloc[i]), "L") for i in piv_low_idx]
    points.sort(key=lambda x: x[0])
    filtered = []
    for pt in points:
        if not filtered:
            filtered.append(pt)
            continue
        last = filtered[-1]
        if pt[2] == last[2]:
            if pt[2] == "H" and pt[1] > last[1]:
                filtered[-1] = pt
            elif pt[2] == "L" and pt[1] < last[1]:
                filtered[-1] = pt
        else:
            filtered.append(pt)
    return filtered


def ratio_in_range(value, lo, hi, tol=HARMONIC_TOLERANCE):
    return (lo - tol) <= value <= (hi + tol)


def detect_harmonic_patterns(alt_points):
    results = []
    if len(alt_points) < 5:
        return results
    X, A, B, C, D = alt_points[-5:]
    xi, xp, xt = X
    ai, ap, at = A
    bi, bp, bt = B
    ci, cp, ct = C
    di, dp, dt = D

    if not (xt == bt == dt and at == ct and xt != at):
        return results

    bullish = xt == "L"  # X,B,D are swing lows -> pattern completes into a bullish reversal at D
    XA, AB, BC, CD, AD = abs(ap - xp), abs(bp - ap), abs(cp - bp), abs(dp - cp), abs(dp - ap)
    if XA == 0 or AB == 0 or BC == 0:
        return results

    ab_xa, bc_ab, cd_bc, ad_xa = AB / XA, BC / AB, CD / BC, AD / XA
    for name, r in HARMONIC_DEFS.items():
        if (ratio_in_range(ab_xa, *r["AB_XA"]) and ratio_in_range(bc_ab, *r["BC_AB"]) and
                ratio_in_range(cd_bc, *r["CD_BC"]) and ratio_in_range(ad_xa, *r["AD_XA"])):
            results.append({
                "pattern": name,
                "bias": "bullish" if bullish else "bearish",
                "D_index": di,
                "D_price": dp,
            })
    return results


## 8. Signal scoring for a single window of bars

In [ ]:
def generate_signal(df, support, resistance, support_line, resistance_line,
                     classic_patterns, harmonic_patterns):
    """Scores the LAST bar of `df` using everything computed from that same window."""
    last = df.iloc[-1]
    n = len(df) - 1
    score = 0
    reasons = []

    if last["EMA9"] > last["EMA21"] > last["EMA50"]:
        score += 2; reasons.append("EMA9 > EMA21 > EMA50 (uptrend)")
    elif last["EMA9"] < last["EMA21"] < last["EMA50"]:
        score -= 2; reasons.append("EMA9 < EMA21 < EMA50 (downtrend)")

    if last["MACD"] > last["MACD_signal"] and last["MACD_hist"] > 0:
        score += 1; reasons.append("MACD bullish crossover")
    elif last["MACD"] < last["MACD_signal"] and last["MACD_hist"] < 0:
        score -= 1; reasons.append("MACD bearish crossover")

    if last["RSI"] < 30:
        score += 1; reasons.append(f"RSI oversold ({last['RSI']:.1f})")
    elif last["RSI"] > 70:
        score -= 1; reasons.append(f"RSI overbought ({last['RSI']:.1f})")

    if last["Close"] > last["VWAP"]:
        score += 1; reasons.append("Price above VWAP")
    else:
        score -= 1; reasons.append("Price below VWAP")

    if last["Close"] <= last["BB_lower"]:
        score += 1; reasons.append("Price at/below lower Bollinger Band")
    elif last["Close"] >= last["BB_upper"]:
        score -= 1; reasons.append("Price at/above upper Bollinger Band")

    for lvl in support:
        if abs(last["Close"] - lvl["price"]) / lvl["price"] <= 0.002:
            score += 1; reasons.append(f"Near support {lvl['price']:.2f}")
    for lvl in resistance:
        if abs(last["Close"] - lvl["price"]) / lvl["price"] <= 0.002:
            score -= 1; reasons.append(f"Near resistance {lvl['price']:.2f}")

    if resistance_line is not None:
        r_val = resistance_line[0] * n + resistance_line[1]
        if last["Close"] > r_val:
            score += 2; reasons.append("Breakout above resistance trend line")
    if support_line is not None:
        s_val = support_line[0] * n + support_line[1]
        if last["Close"] < s_val:
            score -= 2; reasons.append("Breakdown below support trend line")

    for pat in classic_patterns:
        if pat.get("confirmed"):
            if pat["bias"] == "bullish":
                score += 2; reasons.append(f"{pat['pattern']} confirmed (bullish)")
            elif pat["bias"] == "bearish":
                score -= 2; reasons.append(f"{pat['pattern']} confirmed (bearish)")

    for pat in harmonic_patterns:
        if abs(pat["D_index"] - n) <= 3:
            if pat["bias"] == "bullish":
                score += 3; reasons.append(f"{pat['pattern']} harmonic bullish completion at D")
            else:
                score -= 3; reasons.append(f"{pat['pattern']} harmonic bearish completion at D")

    if score >= 4:
        signal = "STRONG BUY"
    elif score >= 2:
        signal = "BUY"
    elif score <= -4:
        signal = "STRONG SELL"
    elif score <= -2:
        signal = "SELL"
    else:
        signal = "HOLD"

    return signal, score, reasons


## 9. Causal, bar-by-bar signal history (this is what drives the arrows)

In [ ]:
def compute_signal_history(df, lookback=CAUSAL_LOOKBACK, warmup=WARMUP_BARS):
    """Walks forward bar by bar. At bar i, only uses rows [i-lookback+1, i] — so a
    signal at bar i never sees data from bar i+1 onward. This is what makes the
    arrows a faithful replay of what the tool would have told you in real time."""
    n = len(df)
    signals = ["HOLD"] * n
    scores = [0] * n
    reasons_list = [[] for _ in range(n)]

    for i in range(warmup, n):
        start = max(0, i - lookback + 1)
        window = df.iloc[start:i + 1]

        piv_high_idx, piv_low_idx = find_pivots(window)
        support, resistance = get_support_resistance(window, piv_high_idx, piv_low_idx)
        support_line, resistance_line = get_trendlines(window, piv_high_idx, piv_low_idx)

        classic_patterns = []
        classic_patterns += detect_double_top_bottom(window, piv_high_idx, piv_low_idx)
        classic_patterns += detect_head_shoulders(window, piv_high_idx, piv_low_idx)
        triangle = detect_triangle(support_line, resistance_line, window["Close"].mean())
        if triangle:
            classic_patterns.append(triangle)

        alt_points = get_alternating_pivots(window, piv_high_idx, piv_low_idx)
        harmonic_patterns = detect_harmonic_patterns(alt_points)

        signal, score, reasons = generate_signal(window, support, resistance, support_line,
                                                   resistance_line, classic_patterns, harmonic_patterns)
        signals[i] = signal
        scores[i] = score
        reasons_list[i] = reasons

    out = df.copy()
    out["Signal"] = signals
    out["Score"] = scores
    out["Reasons"] = reasons_list
    return out


def add_arrow_markers(df):
    """Marks only the bar where a signal *first* turns BUY or SELL, so arrows show
    up once per move instead of on every bar the condition happens to hold."""
    df = df.copy()
    buy_states = {"BUY", "STRONG BUY"}
    sell_states = {"SELL", "STRONG SELL"}
    prev_signal = df["Signal"].shift(1).fillna("HOLD")
    df["BuyArrow"] = df["Signal"].isin(buy_states) & ~prev_signal.isin(buy_states)
    df["SellArrow"] = df["Signal"].isin(sell_states) & ~prev_signal.isin(sell_states)
    return df


## 10. Chart — candlesticks with buy/sell arrows only

In [ ]:
def plot_signals_chart(df, symbol=SYMBOL):
    fig = go.Figure()

    fig.add_trace(go.Candlestick(
        x=df.index, open=df["Open"], high=df["High"], low=df["Low"], close=df["Close"],
        name=symbol, increasing_line_color="#26a69a", decreasing_line_color="#ef5350",
    ))

    buys = df[df["BuyArrow"]]
    sells = df[df["SellArrow"]]

    if len(buys):
        fig.add_trace(go.Scatter(
            x=buys.index, y=buys["Low"] * 0.998, mode="markers", name="BUY",
            marker=dict(symbol="triangle-up", size=16, color="#00e676", line=dict(width=1, color="black")),
            text=[f"BUY  score {s}<br>" + "<br>".join(r) for s, r in zip(buys["Score"], buys["Reasons"])],
            hoverinfo="text+x",
        ))
    if len(sells):
        fig.add_trace(go.Scatter(
            x=sells.index, y=sells["High"] * 1.002, mode="markers", name="SELL",
            marker=dict(symbol="triangle-down", size=16, color="#ff1744", line=dict(width=1, color="black")),
            text=[f"SELL  score {s}<br>" + "<br>".join(r) for s, r in zip(sells["Score"], sells["Reasons"])],
            hoverinfo="text+x",
        ))

    fig.update_layout(
        title=f"{symbol} — 5-Min Candles with Buy/Sell Signals",
        xaxis_rangeslider_visible=False, template="plotly_dark", height=650,
    )
    fig.show()


## 11. Run it — safe to re-run any time during the session

In [ ]:
def run_analysis(symbol=SYMBOL, interval=INTERVAL, period=PERIOD, plot=True):
    raw = fetch_data(symbol, interval, period)
    raw = add_indicators(raw)
    full = compute_signal_history(raw)
    full = add_arrow_markers(full)

    # Chart only today's session — the extra days fetched above exist purely to
    # give the engine enough history for pivots/patterns/trend lines.
    today = full.index[-1].date()
    day_df = full[full.index.date == today]

    last = full.iloc[-1]
    print("=" * 64)
    print(f"{symbol} | {full.index[-1]} | Last Close: {last['Close']:.2f}")
    print(f"SIGNAL: {last['Signal']}  (score: {last['Score']})")
    if last["Reasons"]:
        print("Reasons:")
        for r in last["Reasons"]:
            print("  -", r)
    print("=" * 64)

    if plot:
        plot_signals_chart(day_df, symbol)

    return full


full_history = run_analysis()


## 12. Optional — auto-refresh loop for live intraday monitoring

In [ ]:
import time
from IPython.display import clear_output

def run_live(symbol=SYMBOL, interval=INTERVAL, period=PERIOD, refresh_seconds=300, iterations=100):
    """Re-fetches data, recomputes signals, and redraws the chart every
    `refresh_seconds` (default 5 min), `iterations` times. Keep the Colab tab open
    for this loop to keep running; stop it any time with the cell's stop button."""
    for i in range(iterations):
        clear_output(wait=True)
        try:
            run_analysis(symbol, interval, period, plot=True)
        except Exception as e:
            print("Error:", e)
        time.sleep(refresh_seconds)

# Uncomment to poll every 5 minutes for the rest of the session:
# run_live(refresh_seconds=300, iterations=100)
